# 10 | M5 — Combined 3D U-Net Improvements

**BraTS2020 · hồ sơ nghiên cứu M1–M6 · bản đọc/đối chiếu (clean review edition)**

> **Quan trọng:** Notebook này là bản trình bày có cấu trúc để đọc và review, **không phải Notebook tái chạy toàn bộ**. Mã thực nghiệm nguyên gốc đã được giữ trong `original_notebooks/10-original.ipynb`. Các phần quan trọng được trích nguyên văn vào `scripts/historical_cells/10/`. Không dùng **Run All** trên notebook gốc: một số cell có thể huấn luyện, tạo Dataset hoặc chạy test thật.

- Thí nghiệm: **M5** — Combined optimization.
- Nguồn: `10-m5-combined-improvements.ipynb`; SHA256 của Notebook gốc: `6a13ecc1c16c725b382428108ac81bfb544873e3d33eced32af67bd173b71cc5`.
- Git pin của pipeline dùng chung: `79a95a08f11694278114a89348cf8303a1e39d67` (branch `experiment/unet3d-optimization`).


## 1. Câu hỏi nghiên cứu & thay đổi được kiểm soát

Kết hợp cosine scheduler (M3) + M4a combined loss, từ M1 epoch 10.

**Tập dữ liệu cố định:** BraTS2020, 257 train / 55 internal validation / 56 reserved test, split seed 42. Đầu vào 4 MRI modalities (T1, T1ce, T2, FLAIR), đầu ra 4 logits (background và ba nhãn u đã remap). Train patch `96³`; full-volume inference ROI `96³`, overlap 0.25, Gaussian blending, CPU stitching.

**Riêng M2–M5:** tiếp tục từ M1 epoch 10; kết quả trình bày ở epoch 20.


## 2. Thiết kế kỹ thuật

- Không có test inference; toàn bộ quyết định lựa chọn dựa trên 55 internal validation subjects.
- Code M5.6 training + remote SHA256 backup chính xác nằm trong source cell 09.
- Phân tích sáu mô hình M5.7 từ source cell 10.

**Mã tham chiếu chính:**
- `src/models/m4a_combined_loss.py (extracted)`
- `src/training/engine.py`


## 3. Điểm khởi tạo / API cốt lõi (minh họa, không tự chạy)

Đoạn dưới giúp đọc kiến trúc và khác biệt chính, không tự khởi động training hoặc truy cập test:

```python
# M5 kết hợp hai thay đổi đã cố định trong protocol
from torch.optim.lr_scheduler import CosineAnnealingLR
from src.models.m4a_combined_loss import M4ACombinedLoss, M4A_LOSS_CONFIG
# Mã training/resume lịch sử: scripts/historical_cells/10/...
```


## 4. Kết quả internal validation

**55 bệnh nhân · epoch 20 · macro-average Dice theo từng bệnh nhân.**

| WT | TC | ET | Mean Dice |
|---:|---:|---:|---:|
|0.868796602 | 0.752798564 | 0.678466654 | 0.766687274 |

- M5 Mean Dice = 0.766687274 (validation, epoch 20).
- M5 vs M3 mean delta = +0.003322664; bootstrap CI của chênh trung bình chứa 0.

**Cách đọc:** Dice tốt hơn không đồng nghĩa có ý nghĩa thống kê; so sánh ghép cặp và Holm được lưu trong báo cáo phân tích M5.7 / M6.11.6.


## 5. Training và sao lưu đã được thực hiện như thế nào?

- Các cell huấn luyện gốc được triển khai trên Kaggle và lưu checkpoint/optimizer/AMP (tùy thí nghiệm), không lưu `.pt` trên GitHub.
- Sau khi chạy, các bản backup được xác minh bằng SHA256; kết quả tổng hợp từ những lần chạy đã hoàn tất được bảo quản trên Kaggle.
- Đừng coi đoạn code lưu trong `scripts/historical_cells/` là API thư viện chạy độc lập: chúng phụ thuộc biến trạng thái và mount của Kaggle, cần đưa vào module và kiểm thử trước khi tái sử dụng.
- Các lỗi và bước phục hồi lịch sử được giữ trong notebook gốc nhưng không nằm trong luồng đọc chính.


## 6. Chỉ mục mã nguồn gốc để audit

Những cell sau đã được trích **nguyên văn** vào thư mục `scripts/historical_cells/10/`. Cell index tính từ **0** trong file Notebook gốc.

| Nội dung | Cell index | File gốc đã trích |
|---|---:|---|
| M5 setup | 6 | `10/cell_06_integrated_setup.py` |
| Recovery | 8 | `10/cell_08_cold_start_recovery.py` |
| Training and backup | 9 | `10/cell_09_epoch_training_backup.py` |
| Six-model analysis | 10 | `10/cell_10_six_model_statistics.py` |
| Backup repair | 12 | `10/cell_12_analysis_remote_backup.py` |



## 7. Lưu ý tái lập và giới hạn

- **Đây không phải mã tái huấn luyện đã được kiểm thử sau refactor.** Muốn tái chạy phải xác minh phiên bản thư viện, split, SHA256 checkpoint nguồn, thứ tự thao tác và cơ chế backup.
- Những lần khôi phục phiên Kaggle có thể ảnh hưởng thứ tự RNG của DataLoader; checkpoint tại ranh giới epoch không cam kết bitwise identity.
- Không sử dụng kết quả từ test để điều chỉnh tham số sau khi đã khóa mô hình.
- Các mô hình khác kiến trúc (đặc biệt M6) phải được so sánh cả chi phí tính toán, không chỉ Dice.


In [ ]:
# Read-only: inspect fixed metadata from this review notebook.
# Safe to run locally; does not connect to Kaggle or load MRI.
experiment = 'M5'
validation_mean_dice = 0.766687274
validation_cases = 55
print(experiment, "| validation patients:", validation_cases, "| Mean Dice:", validation_mean_dice)


## 9. Quy trình đưa lên GitHub

1. Giữ `original_notebooks` làm bằng chứng lịch sử; không thay thế/ghi đè bản gốc.
2. Chỉ chuyển module mới vào `src/` sau khi đối chiếu hàm/class nguyên bản và chạy smoke test.
3. Đồng bộ kết quả tổng hợp và checksum; **không commit `.pt`, `.nii`, credential, hoặc dữ liệu bệnh nhân chưa được duyệt công khai**.
4. Tạo PR riêng để review thay đổi trước khi merge vào `main`.
